# Kraków - oferty mieszkań (realne dane)

Notebook korzysta wyłącznie z pliku `Otodom_Flat_Listings.csv`.

Zakres:
- filtrujemy oferty do Krakowa,
- wyciągamy dzielnice i poddzielnice,
- jeśli w ogłoszeniu brakuje lokalizacji, próbujemy znaleźć dzielnicę po nazwie ulicy,
- robimy mapę dzielnic i wizualizacje w stylu Python Graph Gallery.

In [ ]:
# Google Colab: instalacja bibliotek
%pip -q install geopandas folium osmnx geopy squarify

In [ ]:
import re
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import folium
import osmnx as ox
import squarify

from pathlib import Path
from matplotlib.ticker import FuncFormatter
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 120)
print("Importy gotowe")

In [ ]:
# Wczytanie danych (wspiera różne warianty nazwy pliku)
candidates = [
    Path("/content/Otodom_Flat_Listings.csv"),
    Path("/content/otodom_flat_listnings.csv"),
    Path("Otodom_Flat_Listings.csv"),
    Path("otodom_flat_listnings.csv"),
]

csv_path = next((p for p in candidates if p.exists()), None)
if csv_path is None:
    raise FileNotFoundError("Nie znaleziono pliku CSV. Umieść go w /content lub katalogu roboczym.")

df = pd.read_csv(csv_path)
print(f"Wczytano: {csv_path}")
print(f"Liczba rekordów (cała Polska): {len(df):,}")

# Normalizacja kolumn
rename_map = {
    "Title": "title",
    "Price": "price",
    "Location": "location",
    "Surface": "surface",
    "Number_of_Rooms": "rooms",
    "Link": "link",
    "City": "city",
}
df = df.rename(columns=rename_map)

for col in ["title", "price", "location", "surface", "rooms", "link", "city"]:
    if col not in df.columns:
        df[col] = np.nan

df["city_norm"] = df["city"].astype(str).str.lower().str.strip()
df["location_norm"] = df["location"].astype(str).str.lower().str.strip()

# Zostawiamy tylko Kraków (na podstawie city lub location)
krk = df[
    df["city_norm"].str.contains("krak", na=False)
    | df["location_norm"].str.contains("krak", na=False)
].copy()

krk["price"] = pd.to_numeric(krk["price"], errors="coerce")
krk["surface"] = pd.to_numeric(krk["surface"], errors="coerce")
krk["rooms"] = pd.to_numeric(krk["rooms"].astype(str).str.extract(r"(\d+)")[0], errors="coerce")
krk = krk.dropna(subset=["price"]) 
krk = krk[krk["price"].between(50_000, 10_000_000)]

print(f"Oferty w Krakowie: {len(krk):,}")
krk[["title", "price", "location", "city"]].head(5)

In [ ]:
DISTRICTS = [
    "Stare Miasto", "Grzegórzki", "Prądnik Czerwony", "Prądnik Biały", "Krowodrza",
    "Bronowice", "Zwierzyniec", "Dębniki", "Łagiewniki-Borek Fałęcki", "Swoszowice",
    "Podgórze Duchackie", "Bieżanów-Prokocim", "Podgórze", "Czyżyny", "Mistrzejowice",
    "Bieńczyce", "Wzgórza Krzesławickie", "Nowa Huta"
]

district_keywords = {
    d: [d.lower()] for d in DISTRICTS
}
district_keywords["Stare Miasto"] += ["kazimierz", "stradom", "stare miasto"]
district_keywords["Podgórze"] += ["zabłocie", "podgorze"]
district_keywords["Krowodrza"] += ["azory"]
district_keywords["Dębniki"] += ["ruczaj", "debniki"]
district_keywords["Nowa Huta"] += ["nowa huta"]

def extract_street(text: str):
    if not isinstance(text, str):
        return None
    m = re.search(r"(?:ul\.?|al\.?|os\.?)[\s]+([A-ZĄĆĘŁŃÓŚŹŻa-ząćęłńóśźż0-9\- ]{3,})", text)
    return m.group(0).strip() if m else None

def normalize_tokens(location: str):
    if not isinstance(location, str):
        return []
    return [t.strip() for t in location.split(",") if t.strip()]

def guess_district_from_text(text: str):
    if not isinstance(text, str):
        return None
    txt = text.lower()
    for district, keys in district_keywords.items():
        if any(k in txt for k in keys):
            return district
    return None

def extract_subdistrict(location: str, district: str):
    tokens = normalize_tokens(location)
    if not tokens:
        return None
    cleaned = []
    for t in tokens:
        tl = t.lower()
        if "krak" in tl or "małopol" in tl or tl in {"polska", "poland"}:
            continue
        if district and district.lower() in tl:
            continue
        cleaned.append(t)
    return cleaned[0] if cleaned else None

krk["street"] = krk["location"].apply(extract_street)
krk.loc[krk["street"].isna(), "street"] = krk.loc[krk["street"].isna(), "title"].apply(extract_street)

krk["district"] = krk["location"].apply(guess_district_from_text)
krk["subdistrict"] = krk.apply(lambda r: extract_subdistrict(r["location"], r["district"]), axis=1)

# Wymaganie 4: jeśli brak lokalizacji, użyj ulicy z tytułu i geokoduj do dzielnicy
missing_loc = krk[krk["district"].isna() & krk["street"].notna()].copy()
print(f"Brak dzielnicy po samym location: {len(missing_loc)} rekordów")

if len(missing_loc) > 0:
    geolocator = Nominatim(user_agent="krakow_district_lookup_colab")
    geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)
    reverse = RateLimiter(geolocator.reverse, min_delay_seconds=1)

    def district_from_street(street):
        try:
            loc = geocode(f"{street}, Kraków, Polska", addressdetails=True)
            if not loc:
                return None, None
            rev = reverse((loc.latitude, loc.longitude), language="pl", addressdetails=True)
            addr = rev.raw.get("address", {}) if rev else {}
            district_text = " ".join([
                str(addr.get("city_district", "")),
                str(addr.get("suburb", "")),
                str(addr.get("neighbourhood", "")),
            ]).strip()
            district = guess_district_from_text(district_text)
            subdistrict = addr.get("neighbourhood") or addr.get("suburb") or addr.get("quarter")
            return district, subdistrict
        except Exception:
            return None, None

    for idx, row in missing_loc.iterrows():
        d, s = district_from_street(row["street"])
        if d:
            krk.at[idx, "district"] = d
        if s and pd.isna(krk.at[idx, "subdistrict"]):
            krk.at[idx, "subdistrict"] = s

# Domknięcie braków
krk["district"] = krk["district"].fillna("Nieustalona dzielnica")
krk["subdistrict"] = krk["subdistrict"].fillna("Nieustalona poddzielnica")
krk["price_m2"] = (krk["price"] / krk["surface"]).replace([np.inf, -np.inf], np.nan)

krk[["title", "street", "district", "subdistrict", "price", "surface"]].head(10)

In [ ]:
# Granice dzielnic Krakowa (GeoPandas + OSM)
district_gdfs = []
for district in DISTRICTS:
    try:
        q = f"{district}, Kraków, województwo małopolskie, Polska"
        g = ox.geocode_to_gdf(q)
        g["district"] = district
        district_gdfs.append(g[["district", "geometry"]])
    except Exception:
        pass

if not district_gdfs:
    raise RuntimeError("Nie udało się pobrać granic dzielnic z OSM.")

districts_gdf = pd.concat(district_gdfs, ignore_index=True)
districts_gdf = gpd.GeoDataFrame(districts_gdf, geometry="geometry", crs="EPSG:4326")
districts_gdf = districts_gdf.dissolve(by="district", as_index=False)

agg_district = (
    krk.groupby("district", dropna=False)
    .agg(
        listings=("title", "count"),
        median_price=("price", "median"),
        median_price_m2=("price_m2", "median"),
    )
    .reset_index()
)

districts_plot = districts_gdf.merge(agg_district, on="district", how="left")
districts_plot[["listings", "median_price", "median_price_m2"]] = districts_plot[["listings", "median_price", "median_price_m2"]].fillna(0)

print("Dzielnice z geometrią:", len(districts_plot))
districts_plot.head(3)

In [ ]:
# Mapa dzielnic: liczba ofert (choropleth)
fig, ax = plt.subplots(1, 1, figsize=(11, 11))
districts_plot.plot(
    column="listings",
    cmap="YlOrRd",
    linewidth=1,
    edgecolor="white",
    legend=True,
    ax=ax,
    missing_kwds={"color": "lightgrey", "label": "Brak danych"},
)

for _, row in districts_plot.iterrows():
    if row.geometry is not None and not row.geometry.is_empty:
        p = row.geometry.representative_point()
        ax.text(p.x, p.y, f"{row['district']}\n{int(row['listings'])}", fontsize=7, ha="center")

ax.set_title("Kraków - liczba ofert mieszkań wg dzielnic", fontsize=14, weight="bold")
ax.axis("off")
plt.show()

In [ ]:
# Mapa poddzielnic (warstwa punktowa po centroidach dzielnic)
subdistrict_counts = (
    krk.groupby(["district", "subdistrict"], dropna=False)
    .size()
    .reset_index(name="listings")
    .sort_values("listings", ascending=False)
)

centroids = districts_plot.copy()
centroids["geometry"] = centroids.geometry.centroid
centroids = centroids[["district", "geometry"]]

sub_map = subdistrict_counts.merge(centroids, on="district", how="left")
sub_map = gpd.GeoDataFrame(sub_map, geometry="geometry", crs="EPSG:4326")

m = folium.Map(location=[50.0614, 19.9383], zoom_start=11, tiles="cartodbpositron")
for _, row in sub_map.head(120).iterrows():
    if row.geometry is None or row.geometry.is_empty:
        continue
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=max(3, min(14, row["listings"] ** 0.5 * 1.4)),
        color="#1f78b4",
        fill=True,
        fill_opacity=0.6,
        popup=(
            f"Dzielnica: {row['district']}<br>"
            f"Poddzielnica: {row['subdistrict']}<br>"
            f"Liczba ofert: {int(row['listings'])}"
        ),
    ).add_to(m)

m

In [ ]:
# Styl: Python Graph Gallery - lollipop chart (mediana ceny po dzielnicach)
lollipop = (
    krk.groupby("district", dropna=False)["price"]
    .median()
    .sort_values()
    .reset_index(name="median_price")
)

fig, ax = plt.subplots(figsize=(10, 8))
ax.hlines(y=lollipop["district"], xmin=0, xmax=lollipop["median_price"], color="#9ecae1", linewidth=3)
ax.plot(lollipop["median_price"], lollipop["district"], "o", color="#08519c", markersize=8)
ax.set_title("Lollipop: mediana ceny ofert wg dzielnicy", fontsize=14, weight="bold")
ax.set_xlabel("Mediana ceny [PLN]")
ax.set_ylabel("Dzielnica")
ax.xaxis.set_major_formatter(FuncFormatter(lambda x, pos: f"{int(x):,}".replace(",", " ")))
plt.tight_layout()
plt.show()

# Styl: Python Graph Gallery - treemap (udział dzielnic w liczbie ofert)
treemap = (
    krk.groupby("district", dropna=False)
    .size()
    .reset_index(name="listings")
    .sort_values("listings", ascending=False)
)

plt.figure(figsize=(12, 7))
colors = sns.color_palette("Spectral", n_colors=len(treemap))
squarify.plot(
    sizes=treemap["listings"],
    label=[f"{d}\n{n}" for d, n in zip(treemap["district"], treemap["listings"])],
    color=colors,
    alpha=0.9,
    text_kwargs={"fontsize": 9}
)
plt.title("Treemap: udział dzielnic w liczbie ofert", fontsize=14, weight="bold")
plt.axis("off")
plt.show()

In [ ]:
# Top oferty i eksport wyników
summary = (
    krk.groupby(["district", "subdistrict"], dropna=False)
    .agg(
        listings=("title", "count"),
        median_price=("price", "median"),
        median_surface=("surface", "median"),
        median_price_m2=("price_m2", "median"),
    )
    .reset_index()
    .sort_values(["listings", "median_price"], ascending=[False, False])
)

print("Top 20 dzielnica/poddzielnica po liczbie ofert:")
display(summary.head(20))

out_path = Path("/content/krakow_offers_enriched.csv")
krk.to_csv(out_path, index=False)
print(f"Zapisano dane po czyszczeniu: {out_path}")